### SFOS API - IPv4 NAT Rules Management

**Purpose:** Create, read, update, move, and delete IPv4 NAT (Network Address Translation) rules

**Topics:**
- Basic CRUD operations using domain StrEnum objects (`Nat.RULES_IPV4`)
- Source NAT (SNAT / Masquerade) and Destination NAT (DNAT / Port Forwarding) creation
- Pagination and filtering
- Error handling
- Rule positioning and priority ordering (`client.move()`)
- Bulk deletion operations

**Requirements:**
- Environment variables configured (.env file)
- SFOS firewall accessible
- API key with appropriate permissions

---

#### Setup: Import Libraries and Configure Client

In [ ]:
from common import print_data, setup_client

from sfos_api import Nat

# Initialize the SFOS API Client
client = setup_client()
print("✅ SFOS API client initialized")

#### Create IPv4 NAT Rules

Create NAT rules using the `Nat.RULES_IPV4` endpoint object.

NAT rules define address translation policies such as SNAT (Masquerade), DNAT (Server Publishing), or 1:1 NAT.

API Link: [Create an IPv4 rule | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/create-nat-rule-ipv-4/)

In [ ]:
# Create an IPv4 SNAT / Masquerade rule
endpoint = Nat.RULES_IPV4
nat_rule_name = "test-nat-rule-snat"

payload = {
    "name": nat_rule_name,
    "description": "Outbound SNAT masquerade rule created by SFOS API",
    "enabled": True,
    "originalSource": {
        "any": True,
    },
    "translatedSource": {
        "type": "masquerade",
    },
    "originalDestination": {
        "any": True,
    },
    "translatedDestination": {
        "type": "original",
    },
    "originalService": {
        "any": True,
    },
    "translatedService": {
        "type": "original",
    },
    "inboundInterface": {
        "any": True,
    },
    "outboundInterface": {
        "any": True,
    },
}

try:
    response = client.create(endpoint, payload)
    print(f"✅ Created SNAT rule '{nat_rule_name}'")
    print_data(response)
except Exception as e:
    print(f"Error creating SNAT rule: {e}")
    print_data(str(e))

In [ ]:
# Create an IPv4 DNAT (Port Forwarding) rule
endpoint = Nat.RULES_IPV4
dnat_rule_name = "test-nat-rule-dnat"

payload = {
    "name": dnat_rule_name,
    "description": "Inbound DNAT port-forwarding rule created by SFOS API",
    "enabled": True,
    "originalSource": {
        "any": True,
    },
    "translatedSource": {
        "type": "original",
    },
    "originalDestination": {
        "any": True,
    },
    "translatedDestination": {
        "type": "original",
    },
    "originalService": {
        "any": True,
    },
    "translatedService": {
        "type": "original",
    },
}

try:
    response = client.create(endpoint, payload)
    print(f"✅ Created DNAT rule '{dnat_rule_name}'")
    print_data(response)
except Exception as e:
    print(f"Error creating DNAT rule: {e}")
    print_data(str(e))

#### Read IPv4 NAT Rules

Retrieve NAT rule information by ID/name or list all NAT rules.

API Links: [Get all IPv4 rules | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/get-all-nat-rules-ipv-4/) | [Get an IPv4 rule | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/get-nat-rule-ipv-4-by-id-or-name/)

In [ ]:
# Get a specific NAT rule by name
endpoint = Nat.RULES_IPV4
rule_name = "test-nat-rule-snat"

try:
    response = client.read(endpoint, id_or_name=rule_name)
    print(f"✅ Retrieved NAT rule '{rule_name}'")
    print_data(response)
except Exception as e:
    print(f"Error reading NAT rule: {e}")
    print_data(str(e))

In [ ]:
# Get all NAT rules with automatic pagination
endpoint = Nat.RULES_IPV4

try:
    response = client.read(endpoint)

    if "items" in response and response["items"]:
        print(f"✅ Retrieved {len(response['items'])} NAT rules")
        print_data(response["items"], keys=["name", "description", "enabled"])
    else:
        print("No NAT rules returned or empty list")
        print_data(response)
except Exception as e:
    print(f"Error reading NAT rules: {e}")
    print_data(str(e))

In [ ]:
# Get NAT rules with filtering
endpoint = Nat.RULES_IPV4
params = {
    "nameContains": "test-nat-",
    "page": 1,
    "pageSize": 10,
}

try:
    response = client.read(endpoint, params=params, pageTotal=False)

    if "items" in response:
        print(f"✅ Retrieved {len(response['items'])} NAT rules matching filter")
        print_data(response["items"])
    else:
        print("No NAT rules found matching filter")
except Exception as e:
    print(f"Error reading NAT rules: {e}")
    print_data(str(e))

#### Update IPv4 NAT Rule

Modify an existing NAT rule's properties.

API Link: [Update an IPv4 rule | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/update-nat-rule-ipv-4-by-id-or-name/)

In [ ]:
# Update a NAT rule
endpoint = Nat.RULES_IPV4
rule_name = "test-nat-rule-snat"

payload = {
    "description": "Updated SNAT masquerade rule - verified via SFOS API",
}

try:
    response = client.update(endpoint, id_or_name=rule_name, data=payload)
    print(f"✅ Updated NAT rule '{rule_name}'")
    print_data(response)
except Exception as e:
    print(f"Error updating NAT rule: {e}")
    print_data(str(e))

#### Move IPv4 NAT Rule

Reposition a NAT rule within the ordered rulebase using `client.move()`.

API Link: [Move an IPv4 rule | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/move-nat-rule-ipv-4/)

In [ ]:
# Move a NAT rule within the rulebase
endpoint = Nat.RULES_IPV4
rule_name = "test-nat-rule-snat"

move_data = {
    "id": rule_name,
    "position": "top",
}

try:
    response = client.move(endpoint, move_data)
    print(f"✅ Moved NAT rule '{rule_name}' to position 'top'")
    print_data(response)
except Exception as e:
    print(f"Error moving NAT rule: {e}")
    print_data(str(e))

#### Delete IPv4 NAT Rules

Remove NAT rule objects (single or bulk delete).

API Links: [Delete an IPv4 rule | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/delete-nat-rule-ipv-4-by-id-or-name/) | [Bulk delete IPv4 rules | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/bulk-delete-nat-rules-ipv-4/)

In [ ]:
# Delete a single NAT rule
endpoint = Nat.RULES_IPV4
rule_name = "test-nat-rule-snat"

try:
    response = client.delete(endpoint, id_or_name=rule_name)
    print(f"✅ Deleted NAT rule '{rule_name}'")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error deleting NAT rule: {e}")
    print_data(str(e))

In [ ]:
# Bulk delete NAT rules
endpoint = Nat.RULES_IPV4

items_to_delete = [
    {"name": "test-nat-rule-snat"},
    {"name": "test-nat-rule-dnat"},
]

try:
    response = client.delete(endpoint, items=items_to_delete)
    print(f"✅ Bulk deleted {len(items_to_delete)} NAT rules")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error bulk deleting NAT rules: {e}")
    print_data(str(e))

In [ ]:
# Close the client session when done
client.close()
print("✅ Client session closed")